# Yemen cholera, second wave: the extended comparison

This notebook follows on from `yemen_cholera_dpinn_vs_ffpinn_fixed.ipynb`. It reruns the comparison
on the **corrected loader**: the original dropped each governorate's first reporting interval and
anchored the initial condition on the second one, about 7× too high at Al Hudaydah. It then adds
eleven analyses the first study did not have:

| § | analysis | what it answers |
|---|---|---|
| 1 | 10 random seeds per PINN | are between-architecture differences larger than initialisation noise? |
| 2 | rotated 4-fold holdout | which arm generalises, pooled over every interval rather than one split |
| 3 | bands on $\beta_1(t)$, $\beta_2(t)$, $\mathcal R(t)$ | how well each quantity is determined |
| 4 | recovery of known values | given a literature-calibrated truth, what comes back? and do the real-data fits agree with published Yemen estimates? |
| 5 | outside reference for $\mathcal R(t)$ | a model-free Cori/EpiEstim $R_t$ from the same counts |
| 6 | sensitivity | the fixed modelling choices: initial condition, age profile, observable, winsorisation, objective, Fourier order, coupling weight |
| 7 | residual diagnostics | is the Poisson likelihood adequate? |
| 8 | simple baselines | a penalised spline and a plain SIR model |
| 9 | all 21 governorates | the deterministic arm and both baselines everywhere |
| 10 | location-varying $\beta(t,x)$ | does transmission that varies by governorate fit the spatial model better? |
| 11 | observed against fitted surfaces | the whole $I(t,x)$ field, not just per-city curves |

**Scope of the PINN fits.** Each PINN fit costs about 3–4 CPU-minutes. Following the study design
agreed for this extension, the four PINN architectures were run with 10 seeds at five governorates
chosen by total reported cases: the top three (Al Hudaydah, Hajjah, Dhamar), the median governorate
(Al Bayda, 11th of 21) and the smallest (Hadramaut). The deterministic arm and both baselines were
run at all 21. Every fit is a separate result file (`results/<city>/<arm>_s<seed>_f<fold>_<variant>.pkl`),
so the tables below rebuild from disk in seconds; `RUN = True` reruns everything.

Day 0 is 2017-05-22; the record runs to day 272 (2018-02-18). Throughout, $\mathcal R(t)$ is the
model's next-generation number including depletion of susceptibles,
$\mathcal R(t)=\bar S(t)\,\kappa_{EI}J_1[\beta_1(t)+\alpha_0\beta_2(t)J_2]$, so it is directly
comparable with an effective reproduction number $R_t$.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd
from IPython.display import Image, display
import seip, data_io, epiestim, analysis as A
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
FIG = "figures"; os.makedirs(FIG, exist_ok=True)
fmt = lambda df, d=3: df.style.format(precision=d, thousands=",").hide(axis="index")

# RUN = True re-runs every fit (about a day of CPU on 4 cores). Otherwise the
# result files shipped in results/ are loaded.
RUN = False
if RUN:
    import subprocess
    for lst in ("cheap", "pinn"):
        subprocess.run(f"python make_jobs.py {lst} > jobs_{lst}.txt && ./run_queue.sh jobs_{lst}.txt 4",
                       shell=True, check=True)
    subprocess.run("./spatial_chain.sh", shell=True, check=True)

recs, idx = A.load("results")
S = A.load_spatial("results/spatial")
panel = data_io.load_panel("data_real", verbose=False)
FOCUS = A.FOCUS
print(f"{len(recs)} result files; spatial fits: {sorted(S)}")
print(idx.groupby(["arm"]).size().to_dict())

## 0. The five focus governorates

Ranked by total reported cases over the window. Selected: ranks 1–3, rank 11 (the median of 21) and
rank 21. They span almost three orders of magnitude in burden, which matters: Hadramaut's record is
mostly zeros and single-digit counts, the regime where a Poisson likelihood is least informative.

In [ ]:
cases = pd.read_csv("data_real/cases_long.csv")
tot = (cases.groupby("governorate").new_cases.sum().sort_values(ascending=False)
       .rename("total reported cases").reset_index())
tot.insert(0, "rank", np.arange(1, len(tot) + 1))
tot["selected"] = tot.governorate.map(lambda g: A.FOCUS_NOTE.get(g, ""))
display(fmt(tot, 0))

## 1. Ten seeds per architecture

Every PINN arm was refitted from 10 random initialisations on the full record at each focus
governorate, with the training budget unchanged from the original notebook (800 pre-training steps,
1,000 Adam, 120 L-BFGS). The deterministic arm has no seed: its optimiser is started from a grid
search and is deterministic. `MAE_cv` is the seed-to-seed coefficient of variation of the in-sample
error; `R_max_lo`–`R_max_hi` is the range over seeds of the largest $\mathcal R(t)$ in the window.

In [ ]:
seeds = A.seed_table(recs, idx)
ss = A.seed_summary(seeds[seeds.arm.isin(A.PINN + ["det"])])
display(fmt(ss[["city", "arm", "n_seeds", "MAE_mean", "MAE_sd", "MAE_cv", "nRMSE_mean",
                "R_mean", "R_max_med", "R_max_lo", "R_max_hi", "env_mean", "env_sd"]], 4))
A.fig_seed_spread(seeds[seeds.arm.isin(A.PINN)], f"{FIG}/E1_seed_spread.png")
display(Image(f"{FIG}/E1_seed_spread.png"))

*(interpretation written from the results)*

## 2. Fuller holdout: rotated 4-fold interleave

The original study held out one fixed set (every 4th interval). Here the held-out offset rotates
through 0, 1, 2, 3, so **every interval is predicted exactly once by a model that never saw it**
(the first interval stays in training in every fold because the initial condition is anchored on
it). Errors are pooled over the four folds. Each PINN arm was run with 10 seeds per fold; the
spline, SIR and deterministic arms are deterministic. `vs_det` is held-out MAE relative to the
deterministic arm; `degradation` is held-out over in-sample MAE.

In [ ]:
cv = A.cv_table(recs, idx)
cvs = A.cv_summary(cv)
foc = cvs[cvs.city.isin(FOCUS)].copy()
foc["arm"] = pd.Categorical(foc.arm, A.ORDER); foc = foc.sort_values(["city", "arm"])
display(fmt(foc[["city", "arm", "n_seeds", "in_MAE", "out_MAE", "out_MAE_sd",
                 "degradation", "vs_det"]], 4))
A.fig_cv(foc, f"{FIG}/E2_holdout.png")
display(Image(f"{FIG}/E2_holdout.png"))
pv = A.paired_vs_det(cv[cv.city.isin(FOCUS)])
print("Held-out MAE of each PINN seed relative to the deterministic arm "
      "(ratio < 1 = PINN better; one-sample Wilcoxon on log ratios across seeds)")
display(fmt(pv, 3))

*(interpretation written from the results)*

## 3. Uncertainty bands on $\beta_1(t)$, $\beta_2(t)$ and $\mathcal R(t)$

Two kinds of band, labelled in every legend because they are not the same object:

* **PINN arms**: the 2.5–97.5 % range over the 10 seeds. This is **optimisation / initialisation
  uncertainty**: how differently the same architecture answers on the same data. It is not a
  sampling interval.
* **Deterministic arm**: a 95 % **delta-method** band. The Gauss–Newton covariance at the optimum
  is inflated by the fitted dispersion (quasi-Poisson, §7 shows why that inflation is needed) and
  propagated *linearly* to $\log\beta_1$, $\log\beta_2$ and $\log\mathcal R$. Linear propagation is
  essential here: the covariance is near-singular along the direction that trades $\beta_1$ against
  $\beta_2$, and pushing random draws along it through the softplus link produces spurious
  swings in $\mathcal R$ that the linearisation correctly assigns zero first-order variance.

The black curve in the $\mathcal R$ row is the model-free Cori estimate of §5. The table gives the
median band width as the ratio upper/lower: 1 is a perfectly determined curve.

In [ ]:
cori = A.cori_all(panel, panel.names)
cori_short = A.cori_all(panel, panel.names, si=epiestim.SI_SHORT)
A.fig_bands(recs, idx, f"{FIG}/E3a_bands_main.png", arms=("det", "dpinn", "ff"), cori=cori)
display(Image(f"{FIG}/E3a_bands_main.png"))
A.fig_bands(recs, idx, f"{FIG}/E3b_bands_factorial.png", arms=("det", "dpinn_fs", "fdpinn"), cori=cori)
display(Image(f"{FIG}/E3b_bands_factorial.png"))

In [ ]:
# Band widths: median over the window of hi/lo, a scale-free width
rows = []
for c in FOCUS:
    for a in ("det", "sir", *A.PINN):
        for key in ("beta10", "beta20", "R"):
            b = A.band_of(recs, idx, c, a, key)
            if b is None:
                continue
            t, lo, mid, hi, kind = b
            rows.append(dict(city=c, arm=a, quantity=key, band=kind,
                             median_hi_over_lo=float(np.median(hi / np.maximum(lo, 1e-300)))))
bw = pd.DataFrame(rows).pivot_table(index=["city", "arm"], columns="quantity",
                                    values="median_hi_over_lo").reset_index()
display(fmt(bw, 2))

*(interpretation written from the results)*

## 4. Does the method recover known values?

**4a. A literature-calibrated truth.** No published study gives $\beta_1(t)$ or $\beta_2(t)$ for
Yemen, so recovery is tested on synthetic counts whose generating values are known and whose
$\mathcal R(t)$ is set to the published second-wave picture: about 2 at the May 2017 surge
(Camacho et al. 2018 report a median $R_t>2$ in 13 of 23 governorates; an initial-stage estimate
from culture-confirmed cases is 2.2 [2.1, 2.3]), between 1 and 2 through the summer, and below 1
by October 2017. The route split is unknown in the literature, so two opposite truths are used:
20 % and 70 % environmental. Counts are Poisson draws on each focus governorate's **real** bulletin
calendar and population, generated from the same single-location model that is fitted, so the
truth is exactly attainable and any miss is the method's. Three replicate datasets per truth.

In [ ]:
rt = A.recovery_table(recs, idx)
rsum = (rt.groupby(["truth", "arm"])
          .agg(n=("seed", "size"), true_share=("true_share", "first"),
               est_share_mean=("est_share", "mean"), est_share_sd=("est_share", "std"),
               R_relerr=("R_relerr", "median"), beta1_relerr=("beta1_relerr", "median"),
               beta2_relerr=("beta2_relerr", "median"), fit_MAE=("fit_MAE", "mean"),
               truth_MAE=("truth_MAE", "mean")).reset_index())
display(fmt(rsum, 3))
A.fig_recovery(recs, idx, rt, f"{FIG}/E4a_recovery.png")
display(Image(f"{FIG}/E4a_recovery.png"))

**4b. Against published estimates.** The fitted $\mathcal R(t)$ on the real record, averaged over
three windows where published estimates exist, next to the model-free Cori estimate, and the
record's own attack rates against those WHO EMRO reported for the second wave. The attack rates
check the data rather than the method: if they did not match, no fitted quantity could.

In [ ]:
lt = A.literature_table(recs, idx, cori)
display(fmt(lt, 2))
ar = A.attack_rates(panel)
display(fmt(ar[ar.WHO_EMRO_2nd_wave_per_10k.notna() | ar.governorate.isin(FOCUS)], 1))

*(interpretation written from the results)*

## 5. An outside reference for $\mathcal R(t)$

The Cori et al. (2013) estimator used by EpiEstim, implemented in `epiestim.py`. It uses only the
counts and a serial interval, nothing from the SEIP model. Bulletin totals are spread uniformly
across the days of their interval; $R_t$ is estimated over trailing 7-day windows with the EpiEstim
default Gamma(1, 5) prior. Two serial intervals: mean 5.0 d (the value used for the 2017 Yemen
record) and mean 3.7 d (reconstructed from the 1853 Danish epidemics), to show how much the
reference itself depends on that choice.

The table scores each arm's $\mathcal R(t)$ (seed median for PINNs) against the Cori curve: mean
absolute difference, median log-ratio (0 = no bias), the fraction of days inside the Cori 95 %
credible interval, and the rank correlation.

In [ ]:
A.fig_cori(cori, cori_short, f"{FIG}/E5_cori.png")
display(Image(f"{FIG}/E5_cori.png"))
ag = A.cori_agreement(recs, idx, cori)
display(fmt(ag, 3))

*(interpretation written from the results)*

## 6. Sensitivity to the fixed modelling choices

Each row changes one choice and refits on the full record; `dMAE_%` and `dRmean_%` are changes
relative to the baseline fit for that city and arm.

| variant | change |
|---|---|
| `trunc` | initial stocks from the truncated-system occupancy instead of the untruncated identity (1.575× and 1.582× smaller) |
| `age_eq`, `age_ln` | exponential (equilibrium) or log-normal initial age profile instead of flat |
| `onset` | observe infectious onsets $\int\sigma E\,da$ instead of the force of infection |
| `nowinsor` | no winsorisation of the 0.5 % largest rates |
| `wls` | weighted least squares instead of Poisson deviance |
| `M2`, `M6` | Fourier order of $\beta$: 2 or 6 instead of 4 |
| `wren5`, `wren80` | PINN renewal-coupling weight 5 or 80 instead of 20 (PINN arms only, seed 0) |

Diffusion is the remaining fixed choice; it only exists in the spatial model and is swept in §10.

In [ ]:
st = A.sensitivity_table(recs, idx)
cols = ["city", "arm", "variant", "MAE", "dMAE_%", "R_mean", "dRmean_%", "R_min", "R_max", "env_share"]
for arm in ("det", "dpinn", "ff"):
    t = st[st.arm == arm]
    if len(t):
        print(f"--- {A.ARM[arm][0]} ---")
        display(fmt(t[cols].sort_values(["city", "variant"]), 3))
spread = (st.groupby(["arm", "variant"]).agg(dMAE_med=("dMAE_%", "median"),
          dR_med=("dRmean_%", "median"), dR_max=("dRmean_%", lambda v: v.abs().max()))
          .reset_index())
display(fmt(spread, 1))

*(interpretation written from the results)*

## 7. Residual diagnostics

For each full-record fit (seed 0): the Pearson dispersion $\hat\phi=\sum r_i^2/(n-p)$ (1 for a
Poisson model that fits), lag-1 autocorrelation and a Ljung–Box test over 10 lags, the
negative-binomial size $k$ implied by $\mathrm{Var}=\mu+\mu^2/k$, and randomized quantile residuals
(Dunn & Smyth), which are standard normal under a correct model (`rqr_sd` ≈ 1, Shapiro p not small).

In [ ]:
dt, keep = A.diagnostics_table(recs, idx, cities=FOCUS)
dt["arm"] = pd.Categorical(dt.arm, A.ORDER)
display(fmt(dt.sort_values(["city", "arm"]), 3))
A.fig_diagnostics(keep, f"{FIG}/E7_diagnostics.png")
display(Image(f"{FIG}/E7_diagnostics.png"))
dall, _ = A.diagnostics_table(recs, idx, cities=panel.names)
print("all 21 governorates, deterministic arm: dispersion median %.1f (range %.1f-%.1f), "
      "NB size median %.1f"
      % tuple(dall[dall.arm == "det"].dispersion.quantile([.5, 0, 1]).tolist()
              + [dall[dall.arm == "det"].nb_size.median()]))

*(interpretation written from the results)*

## 8. Simple baselines

* **Spline**: Poisson P-spline regression of incidence on time (24 cubic B-splines, smoothing chosen
  by AIC on the training rows). No epidemic mechanism; the floor any curve-fitter reaches.
* **SIR**: $S'=-\beta(t)SI$, $I'=\beta(t)SI-\gamma I$, $\gamma=1/7$ d, $\beta(t)$ an order-4 Fourier
  series, Poisson deviance. The simplest mechanistic model with a reproduction number.

Both run through the same rotated 4-fold holdout as every other arm. Held-out MAE at all 21
governorates:

In [ ]:
b = cvs[cvs.arm.isin(["det", "sir", "spline"])]
piv = b.pivot_table(index="city", columns="arm", values="out_MAE")
piv["sir/det"] = piv["sir"] / piv["det"]; piv["spline/det"] = piv["spline"] / piv["det"]
display(piv.loc[panel.names].style.format("{:.3g}"))
print("median over 21 governorates: SIR/det %.3f, spline/det %.3f; det best at %d, SIR best at %d, "
      "spline best at %d" % (piv["sir/det"].median(), piv["spline/det"].median(),
      (piv[["det", "sir", "spline"]].idxmin(axis=1) == "det").sum(),
      (piv[["det", "sir", "spline"]].idxmin(axis=1) == "sir").sum(),
      (piv[["det", "sir", "spline"]].idxmin(axis=1) == "spline").sum()))

*(interpretation written from the results)*

## 9. All 21 governorates

The deterministic SEIP arm (with its delta-method band) and the SIR baseline at every governorate,
against the Cori reference. Panels are ordered by distance from Sana'a.

In [ ]:
A.fig_all_cities_R(recs, idx, cori, list(panel.names[i] for i in np.argsort(panel.xg)),
                   f"{FIG}/E9_all_cities_R.png")
display(Image(f"{FIG}/E9_all_cities_R.png"))
ag21 = A.cori_agreement(recs, idx, cori, cities=panel.names, arms=("det", "sir"))
display(fmt(ag21.groupby("arm")[["mean_abs_diff", "log_ratio_median", "frac_in_CrI", "spearman"]]
            .median().reset_index(), 3))
r21 = []
for c in panel.names:
    for a in ("det", "sir"):
        ds = A.select(recs, idx, city=c, arm=a, fold=-1, variant="base", truth="")
        if ds:
            r21.append(dict(city=c, arm=a, **A.r_summary(ds[0]), **A.errs(ds[0]["yhat"], ds[0]["y"])))
r21 = pd.DataFrame(r21)
display(fmt(r21[r21.arm == "det"][["city", "MAE", "nRMSE", "R_min", "R_mean", "R_max", "env_share"]], 3))

*(interpretation written from the results)*

## 10. Transmission that varies by location: $\beta(t,x)$

The full $(t,x)$ transport–diffusion model fitted by the deterministic solver to all 21
governorates (2,806 intervals) at once, $x\in[0, 799.4]$ km, $D_1=\dots=D_4$ held fixed:

* **shared**: one $\beta_j(t)$ for the whole country (20 parameters), as in the original §13;
* **smooth**: $\beta_j(t,x)$ with a cubic Legendre surface in $x$ per route plus a
  time×distance interaction (32 parameters);
* **governorate**: $\beta_j(t)\cdot k_g$, one multiplicative offset per governorate common to both
  routes, ridge-shrunk toward 1 and interpolated linearly in $x$ (41 parameters).

AIC is deviance + 2 × parameters. The `shared` and `governorate` models are also refitted at
$D=5$ and $D=250$ km²/day: diffusion is the one modelling choice §6 could not test.

In [ ]:
stab = A.spatial_table(S)
display(fmt(stab, 4))
if "governorate_D50" in S:
    A.fig_spatial_R(S, f"{FIG}/E10_spatial_R.png")
    display(Image(f"{FIG}/E10_spatial_R.png"))
    d = S["governorate_D50"]
    k = pd.DataFrame(dict(governorate=d["names"], x_km=d["xg"],
                          offset=d["theta"][-len(d["names"]):],
                          R_mean=d["R_g"].mean(0), R_max=d["R_g"].max(0))).sort_values("x_km")
    display(fmt(k, 3))

*(interpretation written from the results)*

## 11. Observed against fitted surfaces

The observed incidence field $I(t,x)$ against each fitted field. Rows are governorates ordered by
distance from Sana'a; each bulletin interval is drawn across its own span, **nothing is
interpolated between bulletins**, and the model is drawn as its interval average, the quantity it
was fitted to. The lower row is the signed residual relative to each governorate's mean incidence.
The 3-D view renders the same interval fields. The per-city fields stitch 21 independent fits
together; the spatial fields come from one model of the whole country.

In [ ]:
fields = {"Deterministic, per city": A.city_fields(recs, idx, panel, "det", panel.names)[1],
          "Spline baseline, per city": A.city_fields(recs, idx, panel, "spline", panel.names)[1]}
for key, nm in (("shared_D50", "Spatial, shared beta(t)"),
                ("governorate_D50", "Spatial, beta(t, x)")):
    if key in S:
        fields[nm] = A.spatial_field(S[key], panel)
A.fig_surfaces(panel, fields, f"{FIG}/E11a_surfaces_all.png",
               "All 21 governorates: observed against fitted incidence, rows ordered by "
               "distance from Sana'a")
display(Image(f"{FIG}/E11a_surfaces_all.png"))
A.fig_surfaces_3d(panel, fields, f"{FIG}/E11b_surfaces_3d.png")
display(Image(f"{FIG}/E11b_surfaces_3d.png"))
display(fmt(A.surface_errors(panel, fields), 4))

In [ ]:
# the five focus governorates, where every architecture was fitted
ff = {"Deterministic": fields["Deterministic, per city"]}
for a in ("dpinn", "ff", "dpinn_fs", "fdpinn"):
    ff[A.ARM[a][0] + " (seed median)"] = A.city_fields(recs, idx, panel, a, panel.names)[1]
if "Spatial, beta(t, x)" in fields:
    ff["Spatial, beta(t, x)"] = fields["Spatial, beta(t, x)"]
foc_idx = [panel.names.index(c) for c in FOCUS]
mask = np.isin(np.arange(panel.ng), foc_idx)
sub = {k: np.where(mask[None, :], v, np.nan) for k, v in ff.items()}
A.fig_surfaces(panel, sub, f"{FIG}/E11c_surfaces_focus.png",
               "Five focus governorates: observed against every architecture")
display(Image(f"{FIG}/E11c_surfaces_focus.png"))
display(fmt(A.surface_errors(panel, ff, cities=FOCUS), 4))

*(interpretation written from the results)*

## 12. What the extended analysis shows

*(written from the results)*